In [ ]:
import numpy as np
import pandas as pd

from sklearn import set_config
set_config(display="text")

from sklearn.naive_bayes import MultinomialNB   # 다항 분포 나이브 베이즈 
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.metrics import accuracy_score

# 문제 정의
##### 다항 분포 나이브베이즈 분류 모델을 사용하여, 영화 리뷰에 다항 분포 나이브베이즈 분류 모델을 활용해 영화 리뷰가 긍정인지 부정적인지를 분류해 보겠습니다.

# 데이터 수집

In [2]:
review_list = [
                {'movie_review': 'this is great great movie. I will watch again', 'type': 'positive'},
                {'movie_review': 'I like this movie', 'type': 'positive'},
                {'movie_review': 'amazing movie in this year', 'type': 'positive'},
                {'movie_review': 'cool my boyfriend also said the movie is cool', 'type': 'positive'},
                {'movie_review': 'awesome of the awesome movie ever', 'type': 'positive'},
                {'movie_review': 'shame I wasted money and time', 'type': 'negative'},
                {'movie_review': 'regret on this move. I will never never what movie from this director', 'type': 'negative'},
                {'movie_review': 'I do not like this movie', 'type': 'negative'},
                {'movie_review': 'I do not like actors in this movie', 'type': 'negative'},
                {'movie_review': 'boring boring sleeping movie', 'type': 'negative'}
             ]

df = pd.DataFrame(review_list)
df

,movie_review,type
0,this is great great movie. I will watch again,positive
1,I like this movie,positive
2,amazing movie in this year,positive
3,cool my boyfriend also said the movie is cool,positive
4,awesome of the awesome movie ever,positive
5,shame I wasted money and time,negative
6,regret on this move. I will never never what m...,negative
7,I do not like this movie,negative
8,I do not like actors in this movie,negative
9,boring boring sleeping movie,negative


In [4]:
# 정답 데이터의 숫자 처리
df['label'] = df['type'].map({'positive':1, 'negative':0})
df

,movie_review,type,label
0,this is great great movie. I will watch again,positive,1
1,I like this movie,positive,1
2,amazing movie in this year,positive,1
3,cool my boyfriend also said the movie is cool,positive,1
4,awesome of the awesome movie ever,positive,1
5,shame I wasted money and time,negative,0
6,regret on this move. I will never never what m...,negative,0
7,I do not like this movie,negative,0
8,I do not like actors in this movie,negative,0
9,boring boring sleeping movie,negative,0


In [5]:
df_x = df['movie_review']
df_y = df['label']  # 정답 데이터

In [ ]:
# 전처리 작업
# movie_review 문자열로 제공 : 알고리즘의 입력 데이터로 사용 불가
# sklearn 에서 문자열을 숫자로 변환 CountVectorizer

cv = CountVectorizer()  # binary=False : 단어의 갯수를 가짐
# 1. 모든 단어를 열거한 다음 정렬 한다(fit, 사전을 만듬) 2. 문자열 -> 숫자로 변환(transform) 중첩 데이터는 제외, 입력데이터 크기를 동일하게 맞춤  
x_traincv = cv.fit_transform(df_x)
x_traincv  # 행의 갯수, 단어의 갯


<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 58 stored elements and shape (10, 37)>

In [ ]:
# 모든 단어를 열거한 다음 알파벳 순의 단어로 정렬(단어 사전)

cv.get_feature_names_out()

array(['actors', 'again', 'also', 'amazing', 'and', 'awesome', 'boring',
       'boyfriend', 'cool', 'director', 'do', 'ever', 'from', 'great',
       'in', 'is', 'like', 'money', 'move', 'movie', 'my', 'never', 'not',
       'of', 'on', 'regret', 'said', 'shame', 'sleeping', 'the', 'this',
       'time', 'wasted', 'watch', 'what', 'will', 'year'], dtype=object)

In [ ]:
# toarray()

encoded_input = x_traincv.toarray()  # CountVectorizer(binary=False) 인 경우는 중첩된 단어도 포함
encoded_input


array([[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1],
       [0, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0,
        0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 2,
        0, 0, 1, 1, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
        1, 0, 0, 0, 0, 0, 0, 0

In [12]:
# inverse_transform : 숫자 -> 문자
cv.inverse_transform(encoded_input[[0]])

[array(['again', 'great', 'is', 'movie', 'this', 'watch', 'will'],
       dtype='<U9')]

# 다항 분포 나이브베이즈 분류 모델

In [13]:
# 다항 분호 나이브베이즈 분류기 학습
# 등장하는 단어의 횟수 고려

mnb = MultinomialNB()
y_train = df_y.astype('int')

mnb.fit(x_traincv, y_train) # 학습 데이터, y_train 정답 데이터 -> 예측 모델생성

MultinomialNB()

In [14]:
test_feedback_list = [
                {'movie_review': 'great great great movie ever', 'type': 'positive'},
                {'movie_review': 'I like this amazing movie', 'type': 'positive'},
                {'movie_review': 'my boyfriend said great movie ever', 'type': 'positive'},
                {'movie_review': 'cool cool cool', 'type': 'positive'},
                {'movie_review': 'awesome boyfriend said cool movie ever', 'type': 'positive'},
                {'movie_review': 'shame shame shame', 'type': 'negative'},
                {'movie_review': 'awesome director shame movie boring movie', 'type': 'negative'},
                {'movie_review': 'do not like this movie', 'type': 'negative'},
                {'movie_review': 'I do not like this boring movie', 'type': 'negative'},
                {'movie_review': 'aweful terrible boring movie', 'type': 'negative'}
             ]

test_df = pd.DataFrame(test_feedback_list)
test_df

,movie_review,type
0,great great great movie ever,positive
1,I like this amazing movie,positive
2,my boyfriend said great movie ever,positive
3,cool cool cool,positive
4,awesome boyfriend said cool movie ever,positive
5,shame shame shame,negative
6,awesome director shame movie boring movie,negative
7,do not like this movie,negative
8,I do not like this boring movie,negative
9,aweful terrible boring movie,negative


In [15]:
test_df['label'] = test_df['type'].map({'positive':1, 'negative':0})
test_df

,movie_review,type,label
0,great great great movie ever,positive,1
1,I like this amazing movie,positive,1
2,my boyfriend said great movie ever,positive,1
3,cool cool cool,positive,1
4,awesome boyfriend said cool movie ever,positive,1
5,shame shame shame,negative,0
6,awesome director shame movie boring movie,negative,0
7,do not like this movie,negative,0
8,I do not like this boring movie,negative,0
9,aweful terrible boring movie,negative,0


In [17]:
test_x = test_df['movie_review']
test_y = test_df['label'] # 정답 데이터

In [ ]:
# 테스트를 진행
# 이미 사전이 헉습 데이터에서 만들어 졌으므로 transform() 함수 호출

x_testcv = cv.transform(test_x)
predicted = mnb.predict(x_testcv)  # 예측 모델 생성

### 정확도(accuracy)

In [ ]:
accuracy_score(test_y, predicted)
# 1.0 : 100% 정확도

1.0